# Allen–Cahn: localized model discrepancy and parameter directions

One seed, no noise, **two unknowns** $(\nu,\alpha)$; four known forcing amplitudes.
Methods: **GN-DIST, joint-Adam Bi-HYCO, XPINN-C**. Sensitivities are diagnostics,
not a source of training labels or a way of selecting results with the true parameters.

Extract the whole package and keep the companion `.py` file beside this notebook.
Start with `MODE = "smoke"` for a small functionality check, then use `"pilot"`.
The full reference is generated only once. Read `README.md` before interpreting the tables.

In [9]:
# Optional installation in a new environment (then restart the kernel):
# %pip install numpy scipy pandas matplotlib torch nbformat
from pathlib import Path
from dataclasses import replace
import importlib
import pandas as pd
import AllenCahn_TwoParameter as ac
importlib.reload(ac)

MODE = "pilot"    # change to "smoke" for the functionality check
RUN_GN_DIST = True
RUN_BIHYCO = True
RUN_XPINN_C = True
RUN_SENSITIVITY = True

config = ac.Config(
    mode=MODE, seed=7,
    root="AllenCahn_TwoParameter_Results",
    hyco_rounds=3000, local_steps=2,
    hyco_physical_lr=1e-3, hyco_neural_lr=8e-4,
    ic_weight=0.1, ghost_points=640,
    xp_epochs=10000,
    threads=1, device="cpu", plot_dpi=600,
    run_interaction_sensitivity=True,
    run_local_gn_diagnostic=True,
    refinement_check=True,
)


## 1. Reference, calibration and shared data

The reference uses $192^2$ cells and $\Delta t=0.000525$; inversion uses $64^2$
and $\Delta t=0.0015$. The three omitted Gaussian reaction variations are mainly
in the two quadrants observed by Unit 2. Their amplitude is calibrated to a **12% state
discrepancy**, not a 12% coefficient perturbation. All four source amplitudes are known.
A separate $256^2$ refinement check can be disabled with `refinement_check=False`.

In [10]:
experiment = ac.prepare_experiment(config)
print("Actual configuration:", experiment["config"])
print("Calibration:", experiment["calibration"])
print("Data folder:", experiment["data_folder"])


Calibrating the heterogeneous reference (independent of training)...
Data ID: 6f9204346a6788db; calibrated state mismatch: 12.0000%
Actual configuration: Config(mode='pilot', seed=7, root='AllenCahn_TwoParameter_Results', device='cpu', threads=1, target_discrepancy=0.12, reference_n=192, reference_dt=0.000525, reference_stride=4, inverse_n=64, inverse_dt=0.0015, n_train=1200, n_val=300, n_test=300, hyco_rounds=3000, hyco_min_rounds=1500, hyco_patience=400, hyco_val_every=20, local_steps=2, hyco_width=64, hyco_physical_lr=0.001, hyco_neural_lr=0.0008, interaction_weight=0.18, ramp_rounds=150, ghost_points=640, ic_points=96, ic_weight=0.1, pretrain_steps=1000, pretrain_ic_weight=1.5, fixed_diagnostic_every=100, diagnostic_ghost_points=512, diagnostic_ic_points=256, xp_epochs=10000, xp_min_epochs=5000, xp_patience=2000, xp_val_every=200, xp_warmup=900, xp_ramp_epochs=900, xp_width=64, xp_nn_lr=0.0008, xp_phys_lr=0.0002, xp_collocation=96, xp_observation_batch=96, xp_ic_points=64, xp_bc_po

## 2. GN-DIST

Each holder forms its own normal-equation matrix, vector and scalar loss. The coordinator
combines them. The training residuals/Jacobian rows are not pooled for the update.

In [11]:
if RUN_GN_DIST:
    gn_result = ac.run_method(experiment, "gn_dist")


gn_dist: params=[7.30093439e-02 1.04875558e+02], stop=no_accepted_descent_step, solves=93
 method     e_nu  e_alpha_background  e_gamma_joint  physical_state_error_st  time_seconds
gn_dist 0.026542             0.61347       0.614044                 0.114474     12.972248


## 3. Bi-HYCO: joint local Adam and coordinator averaging

Both holders start each round with the same full parameter pair, take **two joint
updates**, and return both physical and neural parameters. The coordinator averages
physical coefficients in their original coordinates and averages network parameters.
Local Adam moments persist. The two holders use the same learning rates. The
cooperative IC weight is 0.1; the separate initial pretraining IC weight stays 1.5.
The upper limit is 3,000 rounds, with validation-based stopping, not a guarantee that
3,000 rounds will run or that they establish convergence.

In [12]:
if RUN_BIHYCO:
    bihyco_result = ac.run_method(experiment, "bihyco")


Bi-HYCO     1: Vphy=7.6928e-02; Vsyn=9.2457e-02; p=[ 0.055109 50.00117 ]
Bi-HYCO    20: Vphy=6.7936e-02; Vsyn=6.5280e-02; p=[ 0.057346 50.067274]
Bi-HYCO    40: Vphy=5.8227e-02; Vsyn=6.8510e-02; p=[ 0.060068 49.93403 ]
Bi-HYCO    60: Vphy=4.9191e-02; Vsyn=5.8344e-02; p=[ 0.062948 49.893641]
Bi-HYCO    80: Vphy=4.2052e-02; Vsyn=5.0599e-02; p=[ 0.065661 49.78201 ]
Bi-HYCO   100: Vphy=3.6151e-02; Vsyn=4.7604e-02; p=[ 0.068302 50.077321]
Bi-HYCO   120: Vphy=3.2783e-02; Vsyn=4.3491e-02; p=[ 0.07059  49.660898]
Bi-HYCO   140: Vphy=3.0709e-02; Vsyn=3.8340e-02; p=[ 0.072555 49.588969]
Bi-HYCO   160: Vphy=3.0191e-02; Vsyn=3.3922e-02; p=[ 0.073725 49.197169]
Bi-HYCO   180: Vphy=3.0243e-02; Vsyn=3.4131e-02; p=[ 0.074379 48.682494]
Bi-HYCO   200: Vphy=3.0097e-02; Vsyn=3.1550e-02; p=[ 0.075036 48.682376]
Bi-HYCO   220: Vphy=2.9997e-02; Vsyn=3.0034e-02; p=[ 0.075231 48.823789]
Bi-HYCO   240: Vphy=2.9979e-02; Vsyn=2.8368e-02; p=[ 0.075228 48.854494]
Bi-HYCO   260: Vphy=2.9816e-02; Vsyn=2.6886e-02; p=

## 4. XPINN-C

Four quadrant networks, known forcing, two shared positive coefficients. Warm-up has
no PDE residual for 900 epochs, then a 900-epoch ramp. Collocation: 50% uniform,
25% near the known initial interface and 25% near the known sources. The maximum is
10,000 epochs, with both validation histories controlling stopping. Its IC weight
remains 7, because the Bi-HYCO weight study does not tune the XPINN objective.

In [13]:
if RUN_XPINN_C:
    xpinn_result = ac.run_method(experiment, "xpinn_c")


XPINN-C      1: Vphy=7.7382e-02; Vnn=6.9163e-01; p=[ 0.055    50.000003]
XPINN-C    200: Vphy=7.7382e-02; Vnn=2.5255e-01; p=[ 0.055    50.000003]
XPINN-C    400: Vphy=7.7382e-02; Vnn=8.5317e-02; p=[ 0.055    50.000003]
XPINN-C    600: Vphy=7.7382e-02; Vnn=4.3784e-02; p=[ 0.055    50.000003]
XPINN-C    800: Vphy=7.7382e-02; Vnn=3.6070e-02; p=[ 0.055    50.000003]
XPINN-C   1000: Vphy=7.2447e-02; Vnn=2.4755e-02; p=[ 0.056472 48.715258]
XPINN-C   1200: Vphy=6.2172e-02; Vnn=2.0168e-02; p=[ 0.059937 46.11363 ]
XPINN-C   1400: Vphy=5.5307e-02; Vnn=1.9039e-02; p=[ 0.062802 44.045177]
XPINN-C   1600: Vphy=5.1210e-02; Vnn=1.8468e-02; p=[ 0.064987 42.368317]
XPINN-C   1800: Vphy=4.9990e-02; Vnn=1.8729e-02; p=[ 0.066154 41.010567]
XPINN-C   2000: Vphy=5.0330e-02; Vnn=1.6497e-02; p=[ 0.066624 39.921981]
XPINN-C   2200: Vphy=5.3211e-02; Vnn=1.7440e-02; p=[ 0.065965 39.028191]
XPINN-C   2400: Vphy=5.8494e-02; Vnn=1.4833e-02; p=[ 0.064373 38.287601]
XPINN-C   2600: Vphy=6.5997e-02; Vnn=1.5889e-02; p=

## 5. Sensitivities and directions

Compute $A_i=\partial_\rho u_{\mathrm{phy}}(D_i;\rho)$ in
$\rho=(\log\nu,\log\alpha)$ coordinates. Report column norms, singular values,
rank, condition numbers and weakest/strongest right singular vectors for each holder
and their combination. Project **structural discrepancy** and **discretization error**
separately onto those directions, and verify derivatives by halving the finite-difference step.

The optional interaction diagnostic eliminates all neural increments from the local
least-squares model, as in Proposition 3.8. It also reports a version with the actual IC
penalty and several SVD rank tolerances. A negligible or zero gain is a legitimate
result, especially with more neural parameters than finite-sample constraints.
No claim of global identifiability or strict interaction gain is built into the code.

In [14]:
if RUN_SENSITIVITY:
    local_control = ac.run_local_physical_diagnostic(experiment)
    sensitivity_table = ac.run_sensitivity(experiment)
    display(sensitivity_table[sensitivity_table["observation_map"].isin(
        ["unit1_rms", "unit2_rms", "balanced_rms"])])


gn_unit1_diagnostic: params=[ 0.07182872 68.76528217], stop=no_accepted_descent_step, solves=75
Observation sensitivity at: background
Observation sensitivity at: initial
Observation sensitivity at: gn_dist
Observation sensitivity at: bihyco
Observation sensitivity at: xpinn_c
Profiling neural increments for the interaction diagnostic...
Saved figure: directional_sensitivity
Saved figure: singular_values


,point,observation_map,nu,alpha,sigma_max,sigma_min,condition,numerical_rank,column_norm_lognu,column_norm_logalpha,column_cosine,profiled_lognu_norm,profiled_logalpha_norm,weakest_lognu,weakest_logalpha,strongest_lognu,strongest_logalpha
3,background,unit1_rms,0.075000,65.000000,1.020192,0.144934,7.039006,2,1.018986,0.153185,0.320481,0.965240,0.145106,-0.049114,0.998793,0.998793,0.049114
4,background,unit2_rms,0.075000,65.000000,0.876145,0.139735,6.270031,2,0.875676,0.142642,0.198273,0.858291,0.139810,-0.033123,0.999451,0.999451,0.033123
5,background,balanced_rms,0.075000,65.000000,0.950871,0.142552,6.670325,2,0.950037,0.148008,0.265947,0.915824,0.142678,-0.042349,0.999103,0.999103,0.042349
9,initial,unit1_rms,0.055000,50.000000,0.627813,0.214284,2.929818,2,0.622602,0.228983,0.330978,0.587511,0.216077,-0.136787,0.990600,0.990600,0.136787
10,initial,unit2_rms,0.055000,50.000000,0.543527,0.164305,3.308029,2,0.542951,0.166196,0.143365,0.537343,0.164479,-0.048251,0.998835,0.998835,0.048251
11,initial,balanced_rms,0.055000,50.000000,0.586673,0.192500,3.047659,2,0.584136,0.200068,0.257227,0.564480,0.193336,-0.098355,0.995151,0.995151,0.098355
15,gn_dist,unit1_rms,0.073009,104.875558,1.021209,0.119479,8.547215,2,1.021198,0.119576,0.040106,1.020376,0.119480,-0.004761,0.999989,0.999989,0.004761
16,gn_dist,unit2_rms,0.073009,104.875558,0.925987,0.127560,7.259245,2,0.925976,0.127641,-0.035261,0.925400,0.127561,0.004954,0.999988,0.999988,-0.004954
17,gn_dist,balanced_rms,0.073009,104.875558,0.974750,0.123674,7.881640,2,0.974750,0.123674,0.003027,0.974746,0.123674,-0.000390,1.000000,1.000000,0.000390
21,bihyco,unit1_rms,0.072801,71.964793,0.958216,0.142003,6.747877,2,0.957288,0.148128,0.281453,0.918590,0.142140,-0.044487,0.999010,0.999010,0.044487


## 6. Final metrics, saved states, 600-dpi figures and LaTeX tables

Parameter estimates use the physical-validation checkpoint. Neural states use their
own neural-validation checkpoint; paired states are saved as well. XPINN's main
state error is **only the direct neural prediction**, never its auxiliary forward solve.
One seed does not yield a sample standard deviation.

All figures have 16-point fonts, no titles, and 600 dpi. The twelve snapshots display
their times at 16 points. Panel order and checkpoint descriptions are stored separately
for use in captions.

In [15]:
summary = ac.finalize_experiment(experiment, analysis=False)
display(summary[[x for x in ["method", "e_nu", "e_alpha_background", "e_gamma_joint",
    "physical_state_error_st", "neural_state_error_st", "physical_checkpoint",
    "neural_checkpoint", "time_seconds"] if x in summary.columns]])
print("All outputs:", experiment["report"])


Saved figure: reference_12_snapshots
Saved figure: gn_dist_physical_12_snapshots
Saved figure: bihyco_physical_12_snapshots
Saved figure: bihyco_neural_12_snapshots
Saved figure: bihyco_validation
Saved figure: bihyco_nu_trajectory
Saved figure: bihyco_alpha_trajectory
Saved figure: xpinn_c_neural_12_snapshots
Saved figure: xpinn_c_validation
Saved figure: xpinn_c_nu_trajectory
Saved figure: xpinn_c_alpha_trajectory
Saved figure: state_comparison_and_absolute_errors
Report: C:\Users\roberto.morales\Desktop\Allen-Cahn 08\AllenCahn_TwoParameter_Results\pilot\reports\seed7_6f9204346a6788db


,method,e_nu,e_alpha_background,e_gamma_joint,physical_state_error_st,neural_state_error_st,physical_checkpoint,neural_checkpoint,time_seconds
0,gn_dist,0.026542,0.613470,0.614044,0.114474,NaN,9,NaN,12.972248
1,bihyco,0.029324,0.107151,0.111091,0.116226,0.093186,2860,2940.0,10877.843142
2,xpinn_c,0.117953,0.369068,0.387459,NaN,0.072590,1800,9600.0,731.735511


All outputs: C:\Users\roberto.morales\Desktop\Allen-Cahn 08\AllenCahn_TwoParameter_Results\pilot\reports\seed7_6f9204346a6788db


## Optional continuation — not run automatically

To extend the **same** run, preserve the learning rates, data and architecture. The
following commands restore both parameter blocks, Adam moments, RNG states and best
checkpoints. `force_continue=True` intentionally overrides a previous early stop.
Do not extend or select a run merely to obtain a smaller error against known true
parameters. A final multiseed evaluation should use a frozen protocol.

In [16]:
# experiment["config"] = replace(experiment["config"],
#     hyco_rounds=4500, xp_epochs=15000, force_continue=True)
# ac.run_method(experiment, "bihyco")
# ac.run_method(experiment, "xpinn_c")
# summary = ac.finalize_experiment(experiment)
